# Migrating from MLflow to LangSmith


## Migrating Resources

Import existing **datasets, text prompts, and traces with supported feedback**. Evaluation examples below run new experiments; they do not reconstruct historical experiments.

### Contents
- [Run locally](#run-locally)
- [Run on Databricks](#run-on-databricks)
- [Updating Code](#updating-code): tracing, datasets, evaluations, prompts

<a id="run-locally"></a>
### Run locally

From the repository root, install `providers/mlflow/requirements.txt` in your environment. Authenticate with the Databricks CLI and configure LangSmith access locally; keep credentials out of notebook cells and outputs.

```sh
databricks auth login --host https://YOUR-WORKSPACE.cloud.databricks.com --profile mlflow-migration
python migrate.py --provider mlflow --list-experiments --tracking-uri databricks://mlflow-migration
python migrate.py --provider mlflow --experiment EXPERIMENT_ID --tracking-uri databricks://mlflow-migration --output migration-export
python migrate.py --provider mlflow --bundle migration-export/bundle.json --output migration-export --apply
```

The commands list experiments, export/preview one experiment, then import and read back its resources. Configure `LANGSMITH_API_KEY`, `LANGSMITH_WORKSPACE_ID`, and, if needed, `LANGSMITH_ENDPOINT` for the final command. A saved bundle needs no Databricks authentication during import.

If the destination rejects old timestamps, use a separate directory and prefix:

```sh
python migrate.py --provider mlflow --bundle migration-export/bundle.json --output replay-export --prefix mlflow-replay --replay-timestamps --apply
```

Replay changes displayed dates and preserves original timestamps in metadata. Keep the unchanged bundle, timestamp policy, and checkpoint files together for retries.

<a id="run-on-databricks"></a>
### Run on Databricks

Use the same repository scripts from a Databricks notebook kernel with workspace authentication, or import the self-contained `03_databricks_export_migrate.ipynb`. Use tracking URI `databricks`. Configure LangSmith credentials through secret-backed environment variables. UC trace queries require a SQL warehouse; discovery uses the experiment's monitoring warehouse tag when present, otherwise set `MLFLOW_TRACING_SQL_WAREHOUSE_ID`.

Both environments use `providers/mlflow/main.py` and the resource handlers in `data/`. Run `python migrate.py` for the interactive provider selector, or use the explicit commands above. Source setup notebooks are optional test fixtures, not a prerequisite for migration.

### Scope

Datasets and prompts are associated with the imported Agent in agent-based workspaces. General discovery follows run/trace links for UC prompts; unlinked prompts are outside that export. Scorer configurations and historical run metadata are archived, but historical evaluation relationships, live evaluator bindings, review workflows, model deployments, and Insights/Engine setup are not recreated. See [provider details](README.md).


<a id="updating-code"></a>
## Updating Code

As in the Phoenix and Langfuse runbooks, the remaining examples show source instrumentation and the equivalent LangSmith code. They execute new application activity; they do not import historical experiment results. Run from the repository root. All live sections are opt-in.


## 1. Setup

Use a Python 3.10+ notebook kernel. Offline cells use only the standard library. Optional live sections require `mlflow` 3.x, `langsmith`, and (for prompts) `langchain-core` in that kernel. Dependencies are not installed by this notebook.

For live LangSmith cells, configure `LANGSMITH_API_KEY` in the kernel environment before starting. Set `LANGSMITH_ENDPOINT` and `LANGSMITH_WORKSPACE_ID` if your deployment requires them. Keep credentials out of notebook cells and saved outputs.

MLflow uses the configured tracking server when enabled; this notebook defaults to Databricks. For an existing server, configure `MLFLOW_TRACKING_URI` and its authentication through your environment. Prompt registry support depends on the configured backend. All switches below default to off; enabling them writes demo resources to the selected services.

In [ ]:
import os
import json
from copy import deepcopy
from uuid import uuid4

RUN_MLFLOW = False
RUN_LANGSMITH = False
RUN_PROMPT_REGISTRIES = False
EXPORT_HISTORICAL_TRACES = False

SESSION = uuid4().hex[:8]  # Fresh demo resource names on each setup execution.
PROJECT_NAME = f"mlflow-migration-demo-{SESSION}"
DATASET_NAME = f"mlflow-capitals-{SESSION}"
TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "databricks")

if RUN_MLFLOW:
    import mlflow
    mlflow.set_tracking_uri(TRACKING_URI)
    source_experiment = os.getenv("MLFLOW_EXPERIMENT_ID")
    if source_experiment:
        mlflow.set_experiment(experiment_id=source_experiment)
    else:
        mlflow.set_experiment(os.getenv("MLFLOW_EXPERIMENT_NAME", PROJECT_NAME))

if RUN_LANGSMITH:
    if not os.getenv("LANGSMITH_API_KEY"):
        raise RuntimeError("Configure LANGSMITH_API_KEY in the kernel environment.")
    from langsmith import Client, traceable, tracing_context
    ls_client = Client()

print("Ready. Live service sections are opt-in.")

## 2. Updating tracing code

Use the same deterministic application in both systems. The tool lookup becomes a child span/run of the parent application call. A production app can replace the lookup with its own retriever or model.

For model auto-instrumentation, MLflow offers provider-specific autologging such as `mlflow.openai.autolog()`. LangSmith offers `wrap_openai(OpenAI())`. Migrate those calls separately and turn off old autologging if you intend to stop sending traces to MLflow.

In [ ]:
CAPITALS = {"France": "Paris", "Germany": "Berlin", "Japan": "Tokyo"}

def lookup_capital(country: str) -> str:
    return CAPITALS.get(country, "Unknown")

def answer(country: str) -> dict:
    return {"answer": lookup_capital(country)}

assert answer("France") == {"answer": "Paris"}
answer("Japan")

In [ ]:
# Before: MLflow tracing.
if RUN_MLFLOW:
    @mlflow.trace(name="lookup_capital", span_type="TOOL")
    def mlflow_lookup(country: str) -> str:
        return lookup_capital(country)

    @mlflow.trace(name="capital_assistant", span_type="CHAIN")
    def mlflow_app(country: str) -> dict:
        return {"answer": mlflow_lookup(country)}

    mlflow_app("France")

In [ ]:
# After: LangSmith tracing. Context scopes tracing to this example.
if RUN_LANGSMITH:
    @traceable(name="lookup_capital", run_type="tool", client=ls_client)
    def langsmith_lookup(country: str) -> str:
        return lookup_capital(country)

    @traceable(name="capital_assistant", run_type="chain", client=ls_client)
    def langsmith_app(country: str) -> dict:
        return {"answer": langsmith_lookup(country)}

    with tracing_context(enabled=True, project_name=PROJECT_NAME, client=ls_client):
        langsmith_app("France")
    ls_client.flush()

## 3. Converting evaluation data

MLflow GenAI accepts evaluation records with `inputs`, optional predicted `outputs`, and `expectations`. Here, `expectations.expected_response` becomes LangSmith's reference `outputs.answer`.

Replace these synthetic rows with exported evaluation records after checking the schema. This is an explicit mapping for this dataset, not a generic exporter for every MLflow dataset type. Missing references fail validation rather than silently treating predictions as ground truth.

In [ ]:
mlflow_records = [
    {"inputs": {"country": "France"}, "expectations": {"expected_response": "Paris"}},
    {"inputs": {"country": "Germany"}, "expectations": {"expected_response": "Berlin"}},
    {"inputs": {"country": "Japan"}, "expectations": {"expected_response": "Tokyo"}},
]

def to_langsmith_examples(records):
    examples = []
    for index, row in enumerate(records):
        if not isinstance(row.get("inputs"), dict):
            raise ValueError(f"Row {index}: inputs must be a dictionary")
        expected = row.get("expectations", {}).get("expected_response")
        if not isinstance(expected, str) or not expected.strip():
            raise ValueError(f"Row {index}: expected_response must be a nonempty string")
        examples.append({
            "inputs": deepcopy(row["inputs"]),
            "outputs": {"answer": expected},
            "metadata": {"source": "mlflow", "source_row": index},
        })
    return examples

examples = to_langsmith_examples(mlflow_records)
print(json.dumps(examples[0], indent=2))

In [ ]:
# Reuses the completed dataset on rerun in this kernel.
# If upload fails midway, inspect/delete that demo dataset before retrying;
# this small teaching example is not a resumable bulk importer.
if RUN_LANGSMITH and "ls_dataset" not in globals():
    created_dataset = ls_client.create_dataset(
        dataset_name=DATASET_NAME,
        description="Synthetic reference examples migrated from MLflow evaluation records",
    )
    ls_client.create_examples(dataset_id=created_dataset.id, examples=examples)
    ls_dataset = created_dataset
    print(f"Created dataset with {len(examples)} examples.")

## 4. Migrating evaluations

MLflow's prediction function receives the input fields as keyword arguments. LangSmith's target receives one input dictionary. Both examples score the same answer against the same reference. No LLM judge is involved.

Each live evaluation cell creates a new evaluation run/experiment when rerun. These cells evaluate the application again; they do not import old evaluation scores.

In [ ]:
def exact_match(outputs, reference_outputs):
    return outputs["answer"].strip().casefold() == reference_outputs["answer"].strip().casefold()

local_scores = [exact_match(answer(**ex["inputs"]), ex["outputs"]) for ex in examples]
assert all(local_scores)
assert not exact_match({"answer": "Lyon"}, {"answer": "Paris"})
print({"examples": len(local_scores), "accuracy": sum(local_scores) / len(local_scores)})

In [ ]:
if RUN_MLFLOW:
    from mlflow.genai.scorers import scorer

    @scorer
    def mlflow_exact_match(outputs, expectations) -> bool:
        return exact_match(outputs, {"answer": expectations["expected_response"]})

    mlflow_results = mlflow.genai.evaluate(
        data=mlflow_records,
        predict_fn=mlflow_app,
        scorers=[mlflow_exact_match],
    )

In [ ]:
if RUN_LANGSMITH:
    def langsmith_target(inputs: dict) -> dict:
        return langsmith_app(country=inputs["country"])

    def langsmith_exact_match(outputs: dict, reference_outputs: dict) -> dict:
        return {"key": "exact_match", "score": float(exact_match(outputs, reference_outputs))}

    langsmith_results = ls_client.evaluate(
        langsmith_target,
        data=ls_dataset.id,
        evaluators=[langsmith_exact_match],
        experiment_prefix="mlflow-migration",
        max_concurrency=1,
    )

## 5. Migrating prompts

MLflow text prompt templates use double-brace variables. LangChain's default text templates use single braces. Convert a known template explicitly, then compare the rendered text. A global brace replacement can corrupt JSON or literal braces in real prompts.

The live example registers one MLflow text prompt and pushes one equivalent LangSmith prompt. A real migration must enumerate versions, record the source-version → destination-commit mapping, and recreate aliases/tags deliberately. Chat prompts, model configuration, and aliases are outside this text-template example.
For the MLflow prompt example, set `MLFLOW_PROMPT_NAME` to a writable registry name (for UC, `catalog.schema.prompt_name`). The LangSmith-only example needs no MLflow registry.


In [ ]:
mlflow_template = "What is the capital of {{country}}? Reply with only the city."
langsmith_template = "What is the capital of {country}? Reply with only the city."
expected_text = "What is the capital of France? Reply with only the city."
assert langsmith_template.format(country="France") == expected_text

if RUN_PROMPT_REGISTRIES:
    from langchain_core.prompts import PromptTemplate
    prompt_name = f"capital-prompt-{SESSION}"
    if RUN_MLFLOW:
        source_prompt = mlflow.genai.register_prompt(name=os.environ["MLFLOW_PROMPT_NAME"], template=mlflow_template)
        loaded_source = mlflow.genai.load_prompt(f"prompts:/{source_prompt.name}/{source_prompt.version}")
        assert loaded_source.format(country="France") == expected_text
    if RUN_LANGSMITH:
        destination_prompt = PromptTemplate.from_template(langsmith_template)
        destination_url = ls_client.push_prompt(prompt_name, object=destination_prompt, is_public=False)
        restored = ls_client.pull_prompt(prompt_name)
        assert restored.format(country="France") == expected_text
        print("LangSmith prompt rendering verified.")


## 6. Inspecting historical traces (optional, read-only)

Set an explicit source experiment ID from your existing MLflow server. Fetch at most 10 traces for inspection. The default demo experiment is intentionally not used as the source.

The exported records remain in memory. Review and redact real payloads before displaying, saving, or transferring them. This cell does not upload them to LangSmith.

A production importer must preserve parent/child structure, original UTC timestamps, errors, inputs/outputs, and source IDs. Map span types and token usage explicitly; use stable destination IDs and a checkpoint ledger for retries. Translate assessments into feedback separately. Verify counts, nesting, and sample payloads before scaling beyond the sample. See the upstream README's bulk-export guidance.

In [ ]:
SOURCE_EXPERIMENT_ID = os.getenv("MLFLOW_EXPERIMENT_ID", "")
# Enable EXPORT_HISTORICAL_TRACES in setup to read this source experiment.

if EXPORT_HISTORICAL_TRACES:
    import mlflow
    if not SOURCE_EXPERIMENT_ID:
        raise ValueError("Set SOURCE_EXPERIMENT_ID explicitly.")
    mlflow.set_tracking_uri(TRACKING_URI)
    source_traces = mlflow.search_traces(
        locations=[SOURCE_EXPERIMENT_ID],
        max_results=10,
        return_type="list",
    )
    trace_export = [trace.to_dict() for trace in source_traces]
    print(f"Loaded {len(trace_export)} traces into memory; no payloads displayed or uploaded.")

## 7. Validation and next steps

The following offline checks validate reference mapping, reject missing ground truth, and confirm that predicted answers are not copied into reference outputs. Live checks still require your configured MLflow and LangSmith environments.

After enabling live sections, check that each tracing system shows a parent application and child lookup, the dataset has three references, both evaluations score 1.0, and the prompt renders the same text. Demo resources are retained for inspection; remove them in the relevant UI when finished.

**Historical migration:** use `main.py` for export, conversion, checkpoints, and read-back. Alias/history reconstruction, online schedules, deployment migration, and general historical evaluation relationships remain limitations. This runbook exercises newly generated application activity.

References: [upstream runbooks](https://github.com/langchain-ai/langsmith-migrations), [MLflow GenAI](https://mlflow.org/docs/latest/genai/), [LangSmith tracing](https://docs.langchain.com/langsmith/annotate-code), [LangSmith evaluation](https://docs.langchain.com/langsmith/evaluation).

In [ ]:
original = deepcopy(mlflow_records)
prediction_row = {"inputs": {"country": "France"}, "outputs": {"answer": "Lyon"},
                  "expectations": {"expected_response": "Paris"}}
assert to_langsmith_examples([prediction_row])[0]["outputs"] == {"answer": "Paris"}
try:
    to_langsmith_examples([{"inputs": {"country": "France"}, "outputs": {"answer": "Paris"}}])
except ValueError:
    pass
else:
    raise AssertionError("Missing references must fail validation")
assert mlflow_records == original
assert len(examples) == 3
print("Offline validation passed.")

## Agent-based workspace associations
In the new Agents UI, datasets and prompts need the reserved Agent resource association. A generic Application tag is insufficient. Run the following from the repository root after the LangSmith examples. For new agent-native tracing, use `ls.agent_address(agent_id, environment)` instead of `project_name`; do not pass both. See [Agents](https://docs.langchain.com/langsmith/agents) and [agent addressing](https://docs.langchain.com/langsmith/log-traces-to-agent).


In [ ]:
if RUN_LANGSMITH:
    from utils.mlflow_destination import attach_agent_resources
    resources = [{"resource_type": "dataset", "resource_id": str(ls_dataset.id)},
                 {"resource_type": "experiment", "resource_id": str(ls_client.read_project(project_name=langsmith_results.experiment_name).id)}]
    if RUN_PROMPT_REGISTRIES:
        resources.append({"resource_type": "prompt", "resource_id": ls_client.get_prompt(prompt_name).id})
    print(attach_agent_resources(ls_client, PROJECT_NAME, resources))
